# ⚛️ Symbolic Generative LLM - Full Colab Deployment

This notebook provisions a completely fresh Google Colab instance, pulling your repository, building the core Rust mathematical engines from scratch, and orchestrating the Python Streamlit UI via an Ngrok tunnel.

In [ ]:
# 1. Clone the repository
import os
# Replace with your actual repository URL if private/different
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/Relative_Active_Graph.git"

if not os.path.exists("Relative_Active_Graph"):
    !git clone {REPO_URL}
%cd Relative_Active_Graph

In [ ]:
# 2. Install Rust Toolchain and C++ Bindgen Dependencies
!sudo apt-get update -y && sudo apt-get install -y build-essential clang libclang-dev
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ['PATH'] += ":/root/.cargo/bin"

In [ ]:
# 3. Build the Core Rust Z3 constraints and IDA* engines
!cd core && cargo build --release

In [ ]:
# 4. Install Python Backend / Induction Dependencies
!pip install -q pyngrok streamlit
# (Optional) Install any other requirements if you have them:
# !pip install -r requirements.txt

# Grant execution privileges to our launch script just in case
!chmod +x launch_colab.sh

In [ ]:
# 5. Authenticate Ngrok
import getpass

print("Get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken")
os.environ["NGROK_AUTH_TOKEN"] = getpass.getpass("Enter your Ngrok Auth Token: ")
os.environ["NGROK_TUNNEL_ACTIVE"] = "1"

In [ ]:
# 6. Launch Production UI
from pyngrok import ngrok
import time

print("\U0001f680 Starting Streamlit backend...")
get_ipython().system_raw('streamlit run app.py &>/dev/null &')  # Runs Streamlit detached
time.sleep(3)

# Start tunnel and print link
ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])
public_url = ngrok.connect(8501)
print(f"\n\n\u2728 Production UI is live at: {public_url}")